# SEC 8-K 關鍵字篩選與自動分類系統

這套程式碼旨在處理大規模（數十萬筆）的 SEC 8-K HTML 申報文件。它能根據設定好的關鍵字（支援模糊與精確搜尋），自動篩選出相關檔案，將其複製到指定資料夾，並產生詳細的統計報表。

## 1. 核心功能

* **精準關鍵字比對**：使用正規表達式 (Regex) 引擎。
    * **Prefix (模糊)**：如設定 `audit`，可抓到 `auditor`, `auditing`。
    * **Exact (精確)**：如設定 `internal control`，僅抓取該特定片語，排除 `internal controls` (複數) 或其他變體。
* **分批處理 (Batch Processing)**：支援手動指定處理範圍（例如只執行第 0 到 100,000 筆）。
* **自動檔案整理**：命中關鍵字的檔案會自動**複製**到輸出目錄，並依照批次歸檔，方便後續人工檢閱。
* **斷點續跑**：若程式中斷，重跑時會自動跳過已完成的批次 Excel 檔，避免重複作業。

---

## 2. 輸入與輸出 (Input & Output)

### 輸入 (Inputs)

| 項目 | 類型 | 說明 |
| :--- | :--- | :--- |
| **HTML_DIR** | 資料夾 | 存放原始 SEC 8-K `.html` 檔案的路徑。 |
| **keywords.xlsx** | Excel | 關鍵字設定檔。程式初次執行會自動產生，需手動編輯填入欲搜尋的字詞。 |
| **all_files_html_list.txt** | 文字檔 | 這是檔案路徑快取清單，裡面有記錄下載好的html檔案。如果沒有會程式自動產生，目的是為了分批下載。 |
| **User Range** | 互動輸入 | 執行時需輸入 **起始索引 (Start Index)** 與 **結束索引 (End Index)**。 |

### 輸出 (Outputs)

| 項目 | 類型 | 說明 |
| :--- | :--- | :--- |
| **COPY_DIR** | 資料夾 | 存放篩選結果。結構為 `batch_1/`, `batch_2/`...，內含命中關鍵字的原始檔案。 |
| **Excel Reports** | 報表 | 檔名如 `batch_1_results.xlsx`。包含檔名、各關鍵字命中次數、原始路徑。 |
| **Console Log** | 螢幕輸出 | 即時顯示處理進度、複製數量及錯誤訊息。 |

---

## 3. 使用指南 (Step-by-Step)

請依序執行 Jupyter Notebook 中的 7 個區塊 (Cells)：

### 步驟 1：環境與參數設定 (Cell 1)
* 修改 `HTML_DIR`：填入您存放原始檔案的路徑。
* 修改 `COPY_DIR`：填入您希望存放篩選結果的路徑。
* 調整 `MAX_WORKERS`：根據電腦核心數調整（建議 4-8）。

### 步驟 2：設定關鍵字 (Cell 2)
1.  執行此 Cell。
2.  前往程式所在資料夾，打開自動產生的 **`keywords.xlsx`**。
3.  編輯規則：
    * **Keyword**欄：填入單字或片語。
    * **Search_Type**欄：填入 `Prefix` (字首模糊) 或 `Exact` (精確比對)。
4.  存檔並關閉 Excel。

### 步驟 3：建立檔案清單 (Cell 3)
* 執行此 Cell。
* 若是第一次執行，程式會掃描整個資料夾（需耗時數分鐘），並建立 `all_files_html_list.txt`。之後執行則會直接讀取此清單。

### 步驟 4：載入核心函式 (Cell 4 & 5)
* 直接執行這兩個 Cells，將文字清理、正規化與單檔掃描邏輯載入記憶體。

### 步驟 5：選擇處理範圍 (Cell 6A、Cell 6B - 互動區)
* 執行後，程式會顯示總檔案數，並跳出輸入框：
    * `Start Index`: 輸入起始筆數（如 `0`）。
    * `End Index`: 輸入結束筆數（如 `200000`）。
* 程式會鎖定該範圍內的檔案進行處理。

### 步驟 6：執行篩選任務 (Cell 7)
* 執行此 Cell，程式開始運作：
    1.  **讀取**：多執行緒讀取檔案並清理文字。
    2.  **比對**：檢查是否包含 Excel 中的關鍵字。
    3.  **複製**：若命中，將檔案複製到 `COPY_DIR/batch_X`。
    4.  **存檔**：每處理完 `BATCH_SIZE` (預設 5 萬筆)，自動儲存 Excel 報表。

---

### 注意事項
* 若您**新增**了原始檔案，請務必刪除 `all_files_html_list.txt`，讓程式重新掃描。
* `COPY_DIR` 會佔用額外硬碟空間，請確保磁碟空間充足。

  ### `keywords.xlsx` 的長相範例

| | A (Keyword) | B (Search_Type) | C (Note) |
| :--- | :--- | :--- | :--- |
| **1** | **Keyword** | **Search_Type** | **Note** |
| **2** | audit | Prefix | 抓 audit, auditor, auditing... |
| **3** | internal control | Exact | 只抓 internal control 這個詞 |
| **4** | bankruptcy | Exact | 只抓 bankruptcy |

---

### 各欄位詳細說明

這個 Excel 表格**必須**包含這三個標題（Header），程式才能讀得懂：

#### **1. Keyword (關鍵字)**
* **填什麼**：您想要搜尋的英文單字或片語。
* **注意**：
    * 大小寫不影響（程式會自動忽略大小寫）。
    * 可以是單字（如 `audit`）也可以是片語（如 `internal control`）。

#### **2. Search_Type (搜尋模式)**
這是最關鍵的一欄，決定程式「怎麼抓」這個字。目前支援兩種模式：

* **`Prefix` (字首模糊搜尋)**
    * **邏輯**：只要單字是「以這個字開頭」的都抓。
    * **適用**：字根變化多的詞。
    * **範例**：設定 `audit`
        * 抓到：**audit**, **audit**or, **audit**ing, **audit**ed

* **`Exact` (精確搜尋)**
    * **邏輯**：必須「完全符合」這個字或片語，前後不能黏著其他字母。
    * **適用**：專有名詞、不想抓到複數或變體的時候。
    * **範例**：設定 `internal control`
        * 抓到：`internal control`
        * **不**抓到：`internal controls` (多了s), `internal controller` (變體)

#### **3. Note (備註)**
* **填什麼**：給您自己看的筆記。
* **用途**：程式完全**不會讀取**這一欄，您可以隨意寫中文、解釋為什麼要設這個字，方便日後管理。

In [4]:
import os
import re
import time
import shutil
import unicodedata as ud
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd
import chardet
from bs4 import BeautifulSoup

HTML_DIR = Path("/Volumes/One Touch/8k_file_html")
COPY_DIR = Path("/Volumes/One Touch/8k_filtered_files_html")
FILE_LIST_PATH = Path("all_files_html_list.txt")
KEYWORD_CONFIG_PATH = "/Users/wanghao/Downloads/keywords.xlsx"

BATCH_SIZE = 50000
MAX_WORKERS = 4

In [25]:
def load_patterns_from_excel(path):
    df = pd.read_excel(path)
    patterns = []
    print("Loading keyword patterns:")
    for _, row in df.iterrows():
        kw = str(row['Keyword']).strip()
        safe_kw = re.escape(kw).replace(r"\ ", r"\s+")
        search_type = str(row['Search_Type']).strip().lower()

        if search_type == 'prefix':
            regex_str = fr"\b{safe_kw}\w*(?:'s)?"
        elif search_type == 'exact':
            regex_str = fr"\b{safe_kw}\b"
        else:
            regex_str = safe_kw

        patterns.append({
            "keyword": kw,
            "pattern": re.compile(regex_str, re.IGNORECASE)
        })
        print(f"   [{kw}] ({search_type}) -> Regex: {regex_str}")
    return patterns

In [8]:
def get_all_files(html_dir, list_path):
    files = []
    if list_path.exists():
        print(f"Loading file list from {list_path}...")
        with open(list_path, "r", encoding="utf-8") as f:
            for line in f:
                path = Path(line.strip())
                if path.exists():
                    files.append(path)
    
    if not files:
        print(f"Scanning directory {html_dir} (this may take a while)...")
        files = list(html_dir.rglob("*.htm*"))
        print(f"Saving list to {list_path}...")
        with open(list_path, "w", encoding="utf-8") as f:
            for p in files:
                f.write(str(p) + "\n")
                
    return files

In [10]:
def read_text_best_effort(path: Path) -> str:
    try:
        raw = path.read_bytes()
        return raw.decode("utf-8")
    except UnicodeDecodeError:
        enc = chardet.detect(raw).get("encoding") or "latin-1"
        try:
            return raw.decode(enc, errors="replace")
        except:
            return raw.decode("latin-1", errors="replace")

def html_to_text(src_html: str) -> str:
    soup = BeautifulSoup(src_html, "lxml")
    return soup.get_text(separator=" ")

def normalize_text(s: str) -> str:
    s = ud.normalize("NFKC", s)
    s = s.lower()
    s = s.replace("\n", " ").replace("\r", " ").replace("\t", " ")
    _ZW_CHARS = r"\u200b\u200c\u200d\u2060\ufeff"
    s = re.sub(f"[{_ZW_CHARS}]", " ", s)
    s = re.sub(r"[^a-z0-9\s']", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

In [12]:
def scan_file(file_path, patterns):
    try:
        raw = read_text_best_effort(file_path)
        text = html_to_text(raw)
        clean_text = normalize_text(text)

        stats = {}
        total_hits = 0

        for p in patterns:
            matches = p['pattern'].findall(clean_text)
            count = len(matches)
            if count > 0:
                stats[p['keyword']] = count
                total_hits += count

        if total_hits > 0:
            return {
                "filename": file_path.name,
                "file_path": str(file_path),
                "total_hits": total_hits,
                **stats
            }
        return None
    except Exception as e:
        return {"filename": file_path.name, "error": str(e)}

# cell 6A

In [21]:
patterns = load_patterns_from_excel(KEYWORD_CONFIG_PATH)

if 'all_files' not in globals() or not all_files:
    all_files = get_all_files(HTML_DIR, FILE_LIST_PATH)

total_available = len(all_files)
print(f"Total files available: {total_available}")

Loading keyword patterns:
   [audit] (prefix) -> Regex: \baudit\w*
   [auditor] (prefix) -> Regex: \bauditor\w*
   [internal control] (exact) -> Regex: \binternal\s+control\b
Total files available: 690853


# cell 6B

In [40]:
print("-" * 30)
try:
    start_input = input(f"Start Index (default 0): ")
    start_idx = int(start_input) if start_input.strip() else 0
    
    end_input = input(f"End Index (default {total_available}): ")
    end_idx = int(end_input) if end_input.strip() else total_available
except ValueError:
    print("Input error, please enter numbers.")
    start_idx, end_idx = -1, -1

if start_idx < 0 or end_idx > total_available or start_idx >= end_idx:
    print("Invalid range. Please re-run this cell.")
    run_execution = False
else:
    target_files = all_files[start_idx:end_idx]
    target_count = len(target_files)
    print(f"Settings: {start_idx} to {end_idx} (Total {target_count})")
    run_execution = True

------------------------------


Start Index (default 0):  
End Index (default 690853):  10000


Settings: 0 to 10000 (Total 10000)


In [52]:
if run_execution:
    num_sub_batches = (target_count + BATCH_SIZE - 1) // BATCH_SIZE
    
    print(f"Starting execution. Split into {num_sub_batches} batches...\n")

    for i in range(num_sub_batches):
        local_start = i * BATCH_SIZE
        local_end = min((i + 1) * BATCH_SIZE, target_count)
        
        global_start_idx = start_idx + local_start
        
        batch_id = (global_start_idx // BATCH_SIZE) + 1
        
        output_excel = f"html_batch_{batch_id}_results.xlsx"
        
        if os.path.exists(output_excel):
            print(f"HTML_Batch {batch_id} ({output_excel}) exists. Skipping.")
            continue
            
        print(f"\n=== Processing Batch {batch_id} (Global Index {global_start_idx} ~ {start_idx + local_end}) ===")
        
        target_batch_dir = COPY_DIR / f"batch_{batch_id}"
        if not target_batch_dir.exists():
            target_batch_dir.mkdir(parents=True, exist_ok=True)

        current_batch_files = target_files[local_start:local_end]
        batch_results = []
        
        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
            future_to_file = {executor.submit(scan_file, f, patterns): f for f in current_batch_files}
            
            processed = 0
            copied_count = 0
            
            for future in as_completed(future_to_file):
                res = future.result()
                processed += 1
                
                if res and "error" not in res:
                    batch_results.append(res)
                    
                    try:
                        src_path = Path(res["file_path"])
                        dst_path = target_batch_dir / src_path.name
                        shutil.copy2(src_path, dst_path)
                        copied_count += 1
                    except Exception as e:
                        print(f"Copy Error: {src_path} -> {e}")
                
                if processed % 1000 == 0:
                    print(f"\rProgress: {processed}/{len(current_batch_files)} | Copied: {copied_count}", end="")
        
        if batch_results:
            df = pd.DataFrame(batch_results)
            cols = [c for c in df.columns if c != "file_path"] + ["file_path"]
            df = df[cols].fillna(0)
            df.to_excel(output_excel, index=False)
            print(f"\nBatch {batch_id} Done. Report: {output_excel}")
        else:
            pd.DataFrame([{"status": "no_hits"}]).to_excel(output_excel, index=False)
            print(f"\nBatch {batch_id} Done (No hits).")

    print("\nSpecified range completed.")
else:
    print("Please go back to the previous cell to set the range.")

Starting execution. Split into 1 batches...


=== Processing Batch 1 (Global Index 0 ~ 10000) ===


KeyboardInterrupt: 